In [81]:
import re
import shutil
import subprocess
from pathlib import Path
from datetime import datetime
import tempfile
import sys
from zipfile import ZipFile, ZIP_DEFLATED

# === CONFIG ===
NEGOCIOS_DIR = Path(r"C:\Tryd7\workspace\replay\import\WDO&DOL\negocios")
IMPORT_DIR   = Path(r"C:\Tryd7\workspace\replay\import\WDO&DOL")
DEST_ROOT    = Path(r"E:\Mercado BMF&BOVESPA bk\tryd\2019\Extraidos DOLWDO")
SEVEN_ZIP_EXE = r"C:\Program Files\7-Zip\7z.exe"  # ou "7z" se estiver no PATH

# Controle de sobrescrita
OVERWRITE_COPY = False     # copiar por cima?
OVERWRITE_ZIP  = True      # recriar .zip se já existir?

# === CHECKS ===
if not Path(SEVEN_ZIP_EXE).exists():
    print(f"❌ 7z.exe não encontrado em: {SEVEN_ZIP_EXE}")
    sys.exit(1)
if not NEGOCIOS_DIR.exists():
    print(f"❌ Pasta não existe: {NEGOCIOS_DIR}")
    sys.exit(1)
DEST_ROOT.mkdir(parents=True, exist_ok=True)

# Regex
pat_gz    = re.compile(r"^(?P<ymd>\d{8})_(?P<kind>(dol|wdo).*)$", re.IGNORECASE)  # .gz relevantes
pat_repl  = re.compile(r"^tryd_replay_(?P<ymd>\d{8})\.0\.0\.dat$", re.IGNORECASE) # replays

def ymd_to_dash(ymd: str) -> str:
    return datetime.strptime(ymd, "%Y%m%d").strftime("%Y-%m-%d")

def copy2(src: Path, dst: Path) -> bool:
    """Copia src→dst. Retorna True se copiou, False se pulou."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and not OVERWRITE_COPY:
        print(f"↩️  Pulando (já existe): {dst}")
        return False
    shutil.copy2(src, dst)
    print(f"✅ Copiado: {src} → {dst}")
    return True

def zip_then_cleanup(dat_path: Path):
    """Compacta dat_path para .zip na mesma pasta, remove o .dat e .gz da pasta."""
    if not dat_path.exists():
        print(f"⚠️  .dat não encontrado para compactar: {dat_path}")
        return

    zip_path = dat_path.with_suffix(".zip")
    if zip_path.exists():
        if OVERWRITE_ZIP:
            try:
                zip_path.unlink()
                print(f"🧹 Removendo zip existente para recriar: {zip_path}")
            except Exception as e:
                print(f"❌ Não foi possível remover {zip_path}: {e}")
                return
        else:
            print(f"↩️  Zip já existe e OVERWRITE_ZIP=False, mantendo: {zip_path}")
            return

    try:
        # Compactar
        with ZipFile(zip_path, "w", compression=ZIP_DEFLATED, compresslevel=6) as zf:
            zf.write(dat_path, arcname=dat_path.name)
        print(f"🗜️  Compactado: {dat_path.name} → {zip_path.name}")

        # Remover o .dat
        dat_path.unlink()
        print(f"🧹 Removido .dat original: {dat_path}")

        # Remover quaisquer .gz nessa pasta
        for gz in dat_path.parent.glob("*.gz"):
            try:
                gz.unlink()
                print(f"🧹 Removido .gz: {gz.name}")
            except Exception as e:
                print(f"⚠️  Falha ao remover {gz.name}: {e}")

    except Exception as e:
        print(f"❌ Erro ao compactar {dat_path}: {e}")

def extract_gz(gz: Path, out_dir: Path):
    print(f"📦 Extraindo: {gz.name}")
    cmd = [SEVEN_ZIP_EXE, "e", "-y", f"-o{out_dir}", str(gz)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print("❌ Erro 7z:", proc.stderr or proc.stdout)
        raise RuntimeError(f"Falha 7z em {gz}")

def process_gz():
    gz_files = sorted(NEGOCIOS_DIR.glob("*.gz"))
    print(f"🔎 Encontrados {len(gz_files)} .gz em {NEGOCIOS_DIR}\n")

    with tempfile.TemporaryDirectory(prefix="negocios_extract_") as tmpdir:
        TMP = Path(tmpdir)
        print(f"📂 Pasta temporária: {TMP}\n")

        for gz in gz_files:
            m = pat_gz.match(gz.stem)  # sem .gz
            if not m:
                print(f"⚠️  Ignorando (fora do padrão dol/wdo): {gz.name}")
                continue

            ymd = m.group("ymd")
            yyyy_mm_dd = ymd_to_dash(ymd)
            dest_folder = DEST_ROOT / yyyy_mm_dd

            # 1) extrair
            try:
                extract_gz(gz, TMP)
            except Exception as e:
                print(e)
                continue

            # 2) pegar arquivos extraídos que comecem com 'YYYYMMDD_'
            for f in TMP.iterdir():
                if not (f.is_file() and f.name.startswith(ymd + "_")):
                    continue
                suffix = f.name.split("_", 1)[1].lower()
                if suffix.startswith("dol"):
                    norm = f"{ymd}_dol"
                elif suffix.startswith("wdo"):
                    norm = f"{ymd}_wdo"
                else:
                    # ex.: di1f21 ou outros — ignorar
                    continue
                copy2(f, dest_folder / norm)

def process_replays():
    """Copia TODOS os tryd_replay_YYYYMMDD.0.0.dat para suas pastas e, após copiar, zipa e limpa."""
    replay_files = sorted(IMPORT_DIR.glob("tryd_replay_*.0.0.dat"))
    if not replay_files:
        print(f"⚠️  Nenhum replay encontrado em {IMPORT_DIR}")
        return

    print(f"\n🎬 Replays encontrados: {len(replay_files)}")
    for rp in replay_files:
        m = pat_repl.match(rp.name)
        if not m:
            print(f"⚠️  Ignorando replay fora do padrão: {rp.name}")
            continue
        ymd = m.group("ymd")
        yyyy_mm_dd = ymd_to_dash(ymd)
        dest_folder = DEST_ROOT / yyyy_mm_dd
        dst_dat = dest_folder / rp.name

        did_copy = copy2(rp, dst_dat)
        # Só compacta/limpa se de fato copiou (evita apagar um .dat antigo que você quis manter)
        if did_copy:
            zip_then_cleanup(dst_dat)

def main():
    process_gz()
    process_replays()
    print("\n🏁 Finalizado.")

if __name__ == "__main__":
    main()


🔎 Encontrados 3 .gz em C:\Tryd7\workspace\replay\import\WDO&DOL\negocios

📂 Pasta temporária: C:\Users\Aldair\AppData\Local\Temp\negocios_extract_lqhni4je

⚠️  Ignorando (fora do padrão dol/wdo): 20190102_di1f21.gz
📦 Extraindo: 20190102_dolg19.gz
✅ Copiado: C:\Users\Aldair\AppData\Local\Temp\negocios_extract_lqhni4je\20190102_dolg19 → E:\Mercado BMF&BOVESPA bk\tryd\2019\Extraidos DOLWDO\2019-01-02\20190102_dol
📦 Extraindo: 20190102_wdog19.gz
↩️  Pulando (já existe): E:\Mercado BMF&BOVESPA bk\tryd\2019\Extraidos DOLWDO\2019-01-02\20190102_dol
✅ Copiado: C:\Users\Aldair\AppData\Local\Temp\negocios_extract_lqhni4je\20190102_wdog19 → E:\Mercado BMF&BOVESPA bk\tryd\2019\Extraidos DOLWDO\2019-01-02\20190102_wdo

🎬 Replays encontrados: 1
✅ Copiado: C:\Tryd7\workspace\replay\import\WDO&DOL\tryd_replay_20190103.0.0.dat → E:\Mercado BMF&BOVESPA bk\tryd\2019\Extraidos DOLWDO\2019-01-03\tryd_replay_20190103.0.0.dat
🗜️  Compactado: tryd_replay_20190103.0.0.dat → tryd_replay_20190103.0.0.zip
🧹 Remov

In [83]:
import pyautogui
import time

print("Mova o mouse até o ponto desejado (ex: botão Play) e espere 5 segundos...")
time.sleep(5)

pos = pyautogui.position()
print(f"Coordenadas do mouse: {pos}")

Mova o mouse até o ponto desejado (ex: botão Play) e espere 5 segundos...
Coordenadas do mouse: Point(x=2787, y=121)
